# S3.20 - PERMISSIVE, DROPMALFORMED, FAILFAST
**Date completed:** November 2026  
**Status:** Completed \
**Key concept:** The read mode decides what Spark does with malformed rows in a file

In [0]:
# ============================================================
# S3.20 — Cell 2
# CSV Read Modes: PERMISSIVE, DROPMALFORMED, FAILFAST
#
# Goal:
# 1. Run all three CSV read modes against the same source file.
# 2. Compare which rows are kept, dropped, or cause failure.
# 3. Capture the raw malformed record in PERMISSIVE mode.
# 4. Distinguish a bad typed value from an extra CSV token.
# 5. Reconcile predicted results with actual results.
# 6. Reinforce lazy evaluation and action-time failures.
#
# Dataset:
# Controlled 5-row CSV written to a Unity Catalog Volume.
#
# Environment:
# Databricks Serverless
#
# Expected behaviour for this exercise:
#
# PERMISSIVE    → 5 rows
# DROPMALFORMED → 4 rows
# FAILFAST      → ERROR
#
# Important:
#
# Row 3:
# 3,Pune,abc
#
# qty expects IntegerType.
# "abc" cannot be parsed as INT.
# → malformed record
#
# Row 4:
# 4,Chennai,40,EXTRA
#
# Schema only expects:
# id, city, qty
#
# Spark reads:
# 4, Chennai, 40
#
# The additional EXTRA token is discarded for the behaviour
# being demonstrated in this Spark environment.
# ============================================================

import pyspark.sql.functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)


print("=== S3.20: CSV READ MODES ===")
print(f"Spark version: {spark.version}")
print()

# ------------------------------------------------------------
# Step 1 — Create UC Volume and Test Path
#
# We use a Unity Catalog Volume because the Spark executors
# must be able to access the source file.
#
# Avoid driver-local tempfile/file:// paths for this
# Serverless Spark exercise.
# ------------------------------------------------------------

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.s3_20_files
""")

BASE_PATH = (
    "/Volumes/workspace/default/"
    "s3_20_files/s3_20"
)

CSV_PATH = (f"{BASE_PATH}/orders.csv")

dbutils.fs.mkdirs(BASE_PATH)

print(f"Test directory: {BASE_PATH}")

print(f"CSV path      : {CSV_PATH}")

print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Create Controlled CSV Source
#
# Header:
#
# id, city, qty
#
# Row 1:
# valid
#
# Row 2:
# valid
#
# Row 3:
# malformed qty
#
# Row 4:
# extra token
#
# Row 5:
# valid
# ------------------------------------------------------------

lines = [

    "id,city,qty",

    "1,Delhi,10",

    "2,Mumbai,20",

    # Bad IntegerType value
    "3,Pune,abc",

    # Extra CSV token
    "4,Chennai,40,EXTRA",

    "5,Delhi,50",
]

source_rows = (
    len(lines) - 1
)

dbutils.fs.put(
    CSV_PATH,
    "\n".join(lines),True)

print(f"Source data rows: {source_rows}")

print("CSV file created.")

print()

In [0]:
# ------------------------------------------------------------
# Step 3 — Explicit Schema
#
# _corrupt_record is included so PERMISSIVE mode can preserve
# the original malformed CSV line for inspection.
# ------------------------------------------------------------

schema = StructType([

    StructField("id",IntegerType(),True),

    StructField("city", StringType(),True),

    StructField("qty", IntegerType(),True),

    StructField("_corrupt_record", StringType(),True)])

print("=== STEP 3: EXPECTED SCHEMA ===")

print(schema)

print()

In [0]:
# ------------------------------------------------------------
# Step 4 — Reusable CSV Reader
#
# mode:
#
# PERMISSIVE
# DROPMALFORMED
# FAILFAST
#
# columnNameOfCorruptRecord tells Spark where to place the
# original malformed CSV line in PERMISSIVE mode.
# ------------------------------------------------------------

def read_with(mode):

    return (
        spark.read
        .option("header", "true")
        .option("mode",mode)
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(schema)
        .csv(CSV_PATH))

In [0]:
# ------------------------------------------------------------
# Step 5 — Prediction BEFORE Execution
#
# Row 1 → valid
# Row 2 → valid
#
# Row 3:
# qty = abc
# → malformed IntegerType conversion
#
# Row 4:
# 4,Chennai,40,EXTRA
# → first 3 schema fields retained
# → EXTRA discarded
#
# Row 5 → valid
#
#
# Prediction:
#
# PERMISSIVE
# → keep all 5 rows
#
# DROPMALFORMED
# → drop row 3
# → return 4 rows
#
# FAILFAST
# → stop with error when malformed row is evaluated
# ------------------------------------------------------------

predicted = {
    "PERMISSIVE": 5,
    "DROPMALFORMED": 4,
    "FAILFAST": "ERROR"
}


print("=== STEP 5: PREDICTION ===")

for mode, result in predicted.items():

    print(
        f"{mode:<14} → {result}"
    )

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — PERMISSIVE
#
# Expected:
#
# All 5 rows remain.
#
# Row 3:
#
# id              = 3
# city            = Pune
# qty             = NULL
# _corrupt_record = raw malformed line
#
# Row 4:
#
# id              = 4
# city            = Chennai
# qty             = 40
#
# EXTRA is not represented in the supplied schema.
#
#
# We intentionally use collect() here because:
#
# - this controlled dataset has only 5 rows
# - we want Spark to materialize all selected fields
# - using count() alone can interact with CSV column pruning
# ------------------------------------------------------------

actual = {}


print("=== STEP 6: PERMISSIVE ===")


try:

    df_permissive = read_with("PERMISSIVE")

    permissive_rows = (df_permissive.collect())

    actual["PERMISSIVE"] = (len(permissive_rows))

    print(
        f"Rows returned: "
        f"{len(permissive_rows)}")

    for row in permissive_rows:

        print(row)


except Exception as e:

    actual["PERMISSIVE"] = (
        f"ERROR {type(e).__name__}"
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        str(e)[:300]
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 7 — Inspect Row 3 and Row 4 in PERMISSIVE Mode
#
# This is the most important comparison in the lesson.
#
#
# Row 3:
#
# 3,Pune,abc
#
# Expected:
# qty = NULL
# corrupt record populated
#
#
# Row 4:
#
# 4,Chennai,40,EXTRA
#
# Expected:
# qty = 40
# not treated like the invalid integer conversion
# ------------------------------------------------------------

print(
    "=== STEP 7: PERMISSIVE — ROW 3 vs ROW 4 ==="
)


df_permissive.filter(

    F.col(
        "id"
    ).isin(
        3,
        4
    )

).show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 8 — Validate PERMISSIVE Bad Record
#
# We identify the malformed integer conversion:
#
# source record exists
# qty parsing failed
# _corrupt_record populated
# ------------------------------------------------------------

print(
    "=== STEP 8: PERMISSIVE CORRUPT RECORDS ==="
)


df_permissive.filter(

    F.col(
        "_corrupt_record"
    ).isNotNull()

).show(
    truncate=False
)

print()

In [0]:
# ------------------------------------------------------------
# Step 9 — DROPMALFORMED
#
# Expected:
#
# Row 3:
# dropped because qty cannot be parsed as IntegerType.
#
# Row 4:
# retained.
#
# Expected result:
#
# 4 rows
# ------------------------------------------------------------

print(
    "=== STEP 9: DROPMALFORMED ==="
)


try:

    df_drop = read_with(
        "DROPMALFORMED"
    )

    drop_rows = (
        df_drop.collect()
    )

    actual["DROPMALFORMED"] = (
        len(drop_rows)
    )


    print(
        f"Rows returned: "
        f"{len(drop_rows)}"
    )


    for row in drop_rows:

        print(row)


except Exception as e:

    actual["DROPMALFORMED"] = (
        f"ERROR {type(e).__name__}"
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        str(e)[:300]
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 10 — FAILFAST
#
# spark.read.csv(...)
#
# creates a lazy DataFrame.
#
# The malformed data is discovered when an action actually
# executes the CSV parsing.
#
# Therefore collect() belongs inside try/except.
# ------------------------------------------------------------

print(
    "=== STEP 10: FAILFAST ==="
)


try:

    df_failfast = read_with(
        "FAILFAST"
    )


    # ACTION:
    # Forces CSV parsing.
    failfast_rows = (
        df_failfast.collect()
    )


    actual["FAILFAST"] = (
        len(failfast_rows)
    )


    print(
        f"Unexpected success: "
        f"{len(failfast_rows)} rows"
    )


except Exception as e:

    actual["FAILFAST"] = "ERROR"

    print(
        "FAILFAST stopped processing."
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:300]}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Prediction vs Actual
# ------------------------------------------------------------

print(
    "=== STEP 11: PREDICTION vs ACTUAL ==="
)


for mode in predicted:

    predicted_value = (
        predicted[mode]
    )

    actual_value = (
        actual.get(mode)
    )

    match = (
        predicted_value
        ==
        actual_value
    )


    print(
        f"{mode:<14} "
        f"predicted={str(predicted_value):<8} "
        f"actual={str(actual_value):<22} "
        f"match={match}"
    )


print()

In [0]:
# ------------------------------------------------------------
# Step 12 — Validate Prediction vs Actual
#
# Do not stop the notebook immediately if Spark's actual
# behaviour differs from our prediction.
# ------------------------------------------------------------

print("=== STEP 12: VALIDATION ===")

print(f"Source rows         : {source_rows}")
print(f"PERMISSIVE actual   : {actual.get('PERMISSIVE')}")
print(f"DROPMALFORMED actual: {actual.get('DROPMALFORMED')}")
print(f"FAILFAST actual     : {actual.get('FAILFAST')}")
print()


assert source_rows == 5, (
    f"Unexpected source row count: {source_rows}"
)


for mode in predicted:

    expected_value = predicted[mode]
    actual_value = actual.get(mode)

    if expected_value == actual_value:

        print(
            f"PASS | {mode:<14} "
            f"predicted={expected_value} "
            f"actual={actual_value}"
        )

    else:

        print(
            f"CHECK | {mode:<14} "
            f"predicted={expected_value} "
            f"actual={actual_value}"
        )


print()
print("Validation completed.")

In [0]:
# ------------------------------------------------------------
# Step 13 — Reconciliation
# ------------------------------------------------------------

print("=== STEP 13: RECONCILIATION ===")

print(f"Source rows        : {source_rows}")

print(f"PERMISSIVE rows    : "
f"{actual['PERMISSIVE']}")

print(
    f"DROPMALFORMED rows : "
    f"{actual['DROPMALFORMED']}")

print(
    f"FAILFAST           : "
    f"{actual['FAILFAST']}")

print("""
Interpretation:

PERMISSIVE:

5 source rows
→ 5 returned rows

No source row was removed.


DROPMALFORMED:

5 source rows
→ 4 returned rows

1 malformed record was removed.


FAILFAST:

5 source rows
→ execution stopped when malformed input was encountered.
""")

In [0]:
# ------------------------------------------------------------
# Step 14 — Row-by-Row Interpretation
# ------------------------------------------------------------

print(
    "=== STEP 14: ROW-BY-ROW INTERPRETATION ==="
)


print("""
ROW 1
-------------------------
1,Delhi,10

Valid record.

PERMISSIVE:
→ kept

DROPMALFORMED:
→ kept

FAILFAST:
→ valid


ROW 2
-------------------------
2,Mumbai,20

Valid record.

PERMISSIVE:
→ kept

DROPMALFORMED:
→ kept

FAILFAST:
→ valid


ROW 3
-------------------------
3,Pune,abc

Schema expects:

qty = IntegerType

Actual source:

qty = "abc"

PERMISSIVE:

→ row retained
→ qty becomes NULL
→ raw malformed record can be captured
  in _corrupt_record


DROPMALFORMED:

→ complete record dropped


FAILFAST:

→ parser raises an error
→ execution stops


ROW 4
-------------------------
4,Chennai,40,EXTRA

Schema expects:

id
city
qty

Parsed fields:

id   = 4
city = Chennai
qty  = 40

The additional EXTRA token is outside the supplied schema.

For the behaviour demonstrated here:

→ first three schema fields are retained
→ EXTRA is discarded
→ row remains


ROW 5
-------------------------
5,Delhi,50

Valid record.

PERMISSIVE:
→ kept

DROPMALFORMED:
→ kept

FAILFAST:
→ valid
""")

In [0]:
# ------------------------------------------------------------
# Step 15 — Important: Read Mode vs Data Quality
#
# PERMISSIVE does NOT mean the data became valid.
#
# It means Spark tolerated the malformed input.
#
# We still need to inspect:
#
# NULL parsed values
# _corrupt_record
# business DQ rules
#
#
# DROPMALFORMED can remove problematic rows, but that creates
# potential silent data loss unless counts are reconciled.
#
#
# FAILFAST enforces a strict ingestion contract:
#
# malformed input
# → fail the read
# ------------------------------------------------------------

print(
    "=== STEP 15: READ MODE vs DATA QUALITY ==="
)


print("""
PERMISSIVE:

Keep processing
      ↓
Capture / tolerate malformed input
      ↓
Still validate the resulting data


DROPMALFORMED:

Malformed row
      ↓
Drop row
      ↓
Potential data loss
      ↓
Reconcile counts


FAILFAST:

Malformed row
      ↓
Stop execution
      ↓
Pipeline/read failure
""")

In [0]:
# Step 16 — Connection to S3.19
#
# S3.19:
#
# try_cast()
# → value-level safe conversion after data is already in a
#   DataFrame.
#
#
# S3.20:
#
# CSV read mode
# → controls malformed-record behaviour while ingesting
#   the external CSV.
#
#
# These operate at different stages.
# ------------------------------------------------------------

print(
    "=== STEP 16: S3.19 vs S3.20 ==="
)

print("""
S3.19:

STRING column already loaded
        ↓
try_cast()
        ↓
value-level conversion handling

S3.20:

External CSV
        ↓
CSV parser
        ↓
PERMISSIVE / DROPMALFORMED / FAILFAST
        ↓
record-level ingestion behaviour

Easy way to remember:

Read mode
→ What should Spark do with a malformed input record?

try_cast()
→ What should Spark do with a malformed value conversion?
""")

In [0]:
# ------------------------------------------------------------
# Step 17 — Why collect() Is Acceptable Here
#
# Normally:
#
# DataFrame.collect()
#
# must be used carefully because it moves the full result to
# the Driver.
#
# In this controlled exercise:
#
# source_rows = 5
#
# so collect() is intentionally safe and useful.
#
# It also ensures the complete tiny parsed result is evaluated.
#
#
# Do NOT copy this pattern to a production-scale CSV table.
# ------------------------------------------------------------

print(
    "=== STEP 17: collect() SAFETY ==="
)

print("""
This lab:

5 rows
→ collect() is intentionally safe.

Production dataset:

millions of rows
→ do NOT blindly collect().


The function is safe here because we know the exact result size.
""")

In [0]:
# ------------------------------------------------------------
# Step 18 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
PERMISSIVE:

→ Try to retain malformed records.
→ Parsed bad fields can become NULL.
→ Raw malformed record can be captured.
→ Returned row count may remain equal to source count.


DROPMALFORMED:

→ Drop malformed records.
→ Can silently reduce row count.
→ Reconciliation is important.


FAILFAST:

→ Stop when malformed input is encountered.
→ Useful for strict ingestion contracts.


OUR OBSERVED/PREDICTED EXERCISE:

Source rows:
5

PERMISSIVE:
5

DROPMALFORMED:
4

FAILFAST:
ERROR


IMPORTANT:

3,Pune,abc

→ malformed typed value


4,Chennai,40,EXTRA

→ extra token
→ not the same problem as the invalid integer conversion


LAZY EVALUATION:

spark.read.csv(...)
→ creates DataFrame plan

collect()
→ executes parsing

Therefore:

FAILFAST exception
→ surfaces when the parser is actually executed.

DATA-QUALITY PRINCIPLE:

Read mode
≠
complete DQ strategy

Even with PERMISSIVE:

inspect malformed records
       ↓
validate parsed values
       ↓
classify / quarantine if required
       ↓
reconcile counts
       ↓
continue downstream
""")

print("=== S3.20 CELL 2 COMPLETED ===")

# S3.20 — Key Takeaways: CSV Read Modes

## 1. The Three Modes

### PERMISSIVE

Default CSV parsing mode.

```text
Malformed input
      ↓
Keep processing
      ↓
Retain row where possible
```

A field that cannot be parsed into the supplied type can become `NULL`.

If `_corrupt_record` is configured and included in the schema, Spark can preserve the original malformed line for inspection.

---

### DROPMALFORMED

```text
Malformed record
      ↓
Drop record
      ↓
Continue processing
```

No parsing error is raised for that dropped row.

### Important

The output row count can silently shrink.

Therefore reconciliation is important.

---

### FAILFAST

```text
Malformed record encountered during parsing
      ↓
Parsing failure
      ↓
Execution stops
```

Use when malformed input should violate the ingestion contract.

Do not describe this as guaranteeing which physical record is the "first" failure in distributed execution.

---

# 2. Malformed Value vs Extra Token

Our controlled CSV contains:

```text
3,Pune,abc
```

Schema expects:

```text
qty = IntegerType
```

But:

```text
abc
```

cannot be parsed as an integer.

Therefore this is a malformed typed value.

---

We also have:

```text
4,Chennai,40,EXTRA
```

The supplied schema contains only:

```text
id
city
qty
```

The additional token is a different parsing situation from:

```text
abc → IntegerType
```

For the Spark CSV behaviour we expect in this exercise, the first three fields are parsed and the extra token is discarded.

### Important

We will treat this as confirmed only after checking the actual notebook output for row 4.

---

# 3. PERMISSIVE and _corrupt_record

With:

```python
.option(
    "columnNameOfCorruptRecord",
    "_corrupt_record"
)
```

and `_corrupt_record` included in the schema, malformed raw input can be retained for investigation.

Example expected pattern:

```text
id              = 3
city            = Pune
qty             = NULL
_corrupt_record = 3,Pune,abc
```

This is useful for:

```text
inspection
DQ classification
quarantine
RCA
```

---

# 4. Lazy Evaluation

```python
spark.read.csv(...)
```

creates a DataFrame and its execution plan.

The actual parsing work occurs when Spark must materialize the data.

Examples of actions:

```text
collect()
show()
take()
first()
```

Therefore a `FAILFAST` parsing error surfaces when execution reaches the malformed record.

### try/except pattern

```python
try:
    df.collect()

except Exception as e:
    ...
```

The action that triggers the parsing must be inside the `try`.

---

# 5. Reconciliation

Always compare:

```text
source rows
vs
rows returned
```

for ingestion pipelines.

Example:

```text
Source
   ↓
5 rows

PERMISSIVE
   ↓
? rows

DROPMALFORMED
   ↓
? rows

FAILFAST
   ↓
error / no completed result
```

`DROPMALFORMED` especially needs reconciliation because malformed rows can disappear without a parsing exception.

---

# 6. Read Mode Is Not a Full DQ Strategy

Read mode only controls parser behaviour.

A stronger pipeline pattern is:

```text
External file
     ↓
Read mode
     ↓
Parsed DataFrame
     ↓
Inspect malformed records
     ↓
Validate typed values
     ↓
Classify
     ↓
Valid / Quarantine
     ↓
Reconcile
     ↓
Continue downstream
```

PERMISSIVE does **not** mean:

```text
data is valid
```

It means:

```text
Spark tolerated the malformed input
```

---

# 7. S3.19 vs S3.20

### S3.19 — try_cast()

Used after data is already in a DataFrame.

```text
STRING value
    ↓
try_cast()
    ↓
typed value / NULL
```

This is value-level conversion handling.

---

### S3.20 — CSV Read Modes

Used while ingesting the external CSV.

```text
CSV record
    ↓
CSV parser
    ↓
PERMISSIVE / DROPMALFORMED / FAILFAST
```

This is record-level ingestion behaviour.

### Easy way to remember

```text
Read mode
→ What should Spark do with a malformed input record?

try_cast()
→ What should Spark do with a malformed value conversion?
```

---

# 8. Observed in Our Workspace

Not verified yet:

```text
PERMISSIVE row count      = ______

DROPMALFORMED row count   = ______

FAILFAST result           = ______
```

Row 3 in PERMISSIVE:

```text
id              = ______
city            = ______
qty             = ______
_corrupt_record = ______
```

Row 4 in PERMISSIVE:

```text
id              = ______
city            = ______
qty             = ______
_corrupt_record = ______
```

FAILFAST exception type:

```text
________________________
```

These should be filled from the actual notebook output before marking S3.20 complete.

---

# 9. Related Interview Questions and Answers

### Q1. What is the difference between PERMISSIVE and DROPMALFORMED?

PERMISSIVE tries to keep malformed records and can expose the raw malformed line.

DROPMALFORMED removes malformed records from the result.

---

### Q2. Why is DROPMALFORMED risky?

Because malformed rows can disappear without a parsing exception, which can silently reduce record counts.

---

### Q3. What does FAILFAST do?

It fails execution when a malformed record is encountered during parsing.

---

### Q4. Is PERMISSIVE a data-quality solution?

No.

It controls parser behaviour but the resulting records still need validation and DQ handling.

---

### Q5. Why use _corrupt_record?

It preserves the raw malformed input so the bad record can be inspected, classified or quarantined.

---

### Q6. Why must the FAILFAST action be inside try/except?

Because Spark is lazy. The malformed record is discovered when the parsing plan is actually executed.

---

### Q7. What is the difference between S3.19 try_cast() and S3.20 read modes?

`try_cast()` handles individual value conversion inside an already loaded DataFrame.

CSV read modes control how malformed records are handled during ingestion.

---

## 10. Core Understanding

```text
Need to keep malformed records for investigation?
→ PERMISSIVE

Need to discard malformed records?
→ DROPMALFORMED
→ reconcile carefully

Need malformed input to stop ingestion?
→ FAILFAST

Need malformed value conversion after loading?
→ try_cast()

Need reliable pipeline quality?
→ parse + validate + classify + quarantine + reconcile
```

**S3.20 — CSV Read Modes remains open until actual notebook outputs are verified.**